## 1. Dataset Loading and Inspection

In this section, we load the competition dataset and inspect the structure, columns, row counts, missing values, and basic characteristics of the three sources and ground truth.

In [ ]:
import pandas as pd

df = pd.read_csv("some_file.tsv", sep="\t")
df.head()

base = r"E:\Amazon-ML-Challenge-2026\data\student_resource"

train_source1 = pd.read_csv(f"{base}\\dataset\\train\\train_source1.tsv", sep="\t")
train_source2 = pd.read_csv(f"{base}\\dataset\\train\\train_source2.tsv", sep="\t")
train_source3 = pd.read_csv(f"{base}\\dataset\\train\\train_source3.tsv", sep="\t")
train_ground_truth = pd.read_csv(f"{base}\\dataset\\train\\train_ground_truth.tsv", sep="\t")

test_source1 = pd.read_csv(f"{base}\\dataset\\test\\test_source1.tsv", sep="\t")
test_source2 = pd.read_csv(f"{base}\\dataset\\test\\test_source2.tsv", sep="\t")
test_source3 = pd.read_csv(f"{base}\\dataset\\test\\test_source3.tsv", sep="\t")

print("Loaded OK")

Loaded OK


In [2]:
for name, df in [
    ("train_source1", train_source1),
    ("train_source2", train_source2),
    ("train_source3", train_source3),
    ("train_ground_truth", train_ground_truth),
    ("test_source1", test_source1),
    ("test_source2", test_source2),
    ("test_source3", test_source3),
]:
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("missing values:\n", df.isna().sum())


=== train_source1 ===
shape: (2206821, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

=== train_source2 ===
shape: (5034616, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

=== train_source3 ===
shape: (5285603, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

=== train_ground_truth ===
shape: (2206821, 2)
columns: ['source1_entity_id', 'matched_entity_ids']
missing values:
 source1_entity_id          0
matched_entity_ids    123247
dtype: int64

=== test_source1 ===
shape: (1732544, 4)
columns: ['entity_id', 'business_

## 2. Ground Truth and Dataset Distribution

Before designing the matching system, we inspect the distribution of
matches and countries in the training data.

In [3]:
train_source1.head(5)

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [4]:
train_source2.head(5)

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


In [5]:
train_ground_truth.head(10)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [6]:
# Confirm the missing ground truth rows really are "no match" (empty string vs NaN)
print(train_ground_truth['matched_entity_ids'].isna().sum())
print((train_ground_truth['matched_entity_ids'] == '').sum())

# How many matches does a typical Source 1 entity have? (only for non-missing rows)
match_counts = train_ground_truth['matched_entity_ids'].dropna().apply(lambda x: len(x.split(',')))
print(match_counts.describe())

# Country distribution across sources
print(train_source1['country'].value_counts())
print(train_source2['country'].value_counts())

123247
0
count    2.083574e+06
mean     3.665992e+00
std      1.526294e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: matched_entity_ids, dtype: float64
country
US       1323633
India     883188
Name: count, dtype: int64
country
US       3016817
India    2017799
Name: count, dtype: int64


In [7]:
import re

def has_devanagari(text):
    if pd.isna(text):
        return False
    return bool(re.search(r'[\u0900-\u097F]', text))

for name, df in [
    ("train_source1", train_source1),
    ("train_source2", train_source2),
    ("train_source3", train_source3)
]:
    n = df['business_name'].apply(has_devanagari).sum()
    print(f"{name}: {n} / {len(df)} names contain Devanagari script ({n/len(df)*100:.2f}%)")

train_source1: 0 / 2206821 names contain Devanagari script (0.00%)
train_source2: 269424 / 5034616 names contain Devanagari script (5.35%)
train_source3: 158003 / 5285603 names contain Devanagari script (2.99%)


## 3. Train-Validation Split

The training data contains Source 1 records together with their ground-truth
matches from Source 2 and Source 3.

To evaluate our matching approach before applying it to the unseen test data,
we split the Source 1 entities into two groups:

- **80% training/development set** — used to develop the baseline approach.
- **20% validation set** — used to evaluate the approach on unseen Source 1 entities.

The split is random and uses a fixed random seed (`random_state=42`) so that
the same validation set can be reproduced across experiments.

Source 2 and Source 3 are not split at this stage because they serve as the
candidate databases against which Source 1 entities are matched.

In [10]:
from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    train_source1['entity_id'],
    test_size=0.20,
    random_state=42
)

print("Training Source 1 entities:", len(train_ids))
print("Validation Source 1 entities:", len(val_ids))

# Sanity check: no overlap, and counts add up
assert len(set(train_ids) & set(val_ids)) == 0
assert len(train_ids) + len(val_ids) == len(train_source1)
print("Split check passed.")

Training Source 1 entities: 1765456
Validation Source 1 entities: 441365
Split check passed.


In [11]:
train_source1_split = train_source1[
    train_source1['entity_id'].isin(train_ids)
].copy().reset_index(drop=True)

val_source1 = train_source1[
    train_source1['entity_id'].isin(val_ids)
].copy().reset_index(drop=True)

train_ground_truth_split = train_ground_truth[
    train_ground_truth['source1_entity_id'].isin(train_ids)
].copy().reset_index(drop=True)

val_ground_truth = train_ground_truth[
    train_ground_truth['source1_entity_id'].isin(val_ids)
].copy().reset_index(drop=True)

print("Train S1:", train_source1_split.shape)
print("Validation S1:", val_source1.shape)
print("Train ground truth:", train_ground_truth_split.shape)
print("Validation ground truth:", val_ground_truth.shape)

Train S1: (1765456, 4)
Validation S1: (441365, 4)
Train ground truth: (1765456, 2)
Validation ground truth: (441365, 2)


## 4. Baseline Matching Approach

We now develop a simple entity-resolution baseline using the training data
and evaluate its predictions on the held-out validation set.

The baseline will first focus on high-confidence matching rules and
candidate generation before introducing more complex matching methods.

### 4.1 Validation Evaluation

The competition evaluates predictions using macro-averaged F₀.₅ across
Source 1 entities. Since precision is weighted more heavily than recall,
the evaluation function measures the overlap between the predicted and
ground-truth match sets for each Source 1 entity.

This function will be used to evaluate our baseline on the held-out
validation set before generating test predictions.

In [12]:
def parse_ids(cell):
    """Convert a comma-separated ID string (or NaN) into a set of IDs."""
    if pd.isna(cell) or cell == '':
        return set()
    return set(cell.split(','))


def f_beta_score(true_set, pred_set, beta=0.5):
    """Compute F_beta for one entity given its true and predicted match sets."""
    # Singleton case: no true matches
    if len(true_set) == 0:
        return 1.0 if len(pred_set) == 0 else 0.0

    # We have true matches but predicted nothing
    if len(pred_set) == 0:
        return 0.0

    tp = len(true_set & pred_set)
    precision = tp / len(pred_set)
    recall = tp / len(true_set)

    if precision == 0 and recall == 0:
        return 0.0

    beta_sq = beta ** 2
    return (1 + beta_sq) * precision * recall / (beta_sq * precision + recall)


def evaluate_predictions(ground_truth_df, predictions_df):
    """
    ground_truth_df: columns [source1_entity_id, matched_entity_ids]
    predictions_df:  columns [source1_entity_id, matched_entity_ids]
    Returns macro-averaged F_0.5 across all entities in ground_truth_df.
    """
    gt = ground_truth_df.set_index('source1_entity_id')['matched_entity_ids']
    pred = predictions_df.set_index('source1_entity_id')['matched_entity_ids']

    scores = []
    for entity_id in gt.index:
        true_set = parse_ids(gt.loc[entity_id])
        # If we didn't predict anything for this entity at all, treat as empty
        pred_cell = pred.loc[entity_id] if entity_id in pred.index else None
        pred_set = parse_ids(pred_cell)
        scores.append(f_beta_score(true_set, pred_set))

    return sum(scores) / len(scores)

In [13]:
import pandas as pd

test_gt = pd.DataFrame({
    'source1_entity_id': ['S1-00001'],
    'matched_entity_ids': ['S2-00047,S3-00812']
})
test_pred = pd.DataFrame({
    'source1_entity_id': ['S1-00001'],
    'matched_entity_ids': ['S2-00047,S2-00193,S3-00812']
})

score = evaluate_predictions(test_gt, test_pred)
print(score)  # PDF says this should be 0.714

0.7142857142857142


## Evaluation Function — Verification

Before using the evaluation function on our real validation data, we tested it
with a small manually constructed example.

The ground truth contained two correct matches, while the prediction contained
those two matches plus one incorrect match. The resulting F_0.5 score was
approximately 0.7143, confirming that the evaluation function penalizes the
extra false match while giving credit for the two correct matches.

This is only a test of the evaluation code. No model has been trained and no
real validation score has been calculated yet.

### 4.2 Name Normalization for Blocking

Before generating candidate pairs, we first normalize business names so that
records representing the same business are less likely to be separated by
simple formatting differences.

The baseline normalization performs four operations:

- Converts the business name to lowercase.
- Replaces punctuation with spaces.
- Removes selected common legal suffixes such as `Inc`, `Ltd`, `LLC`, and
  `Corporation`.
- Collapses whitespace and reconstructs the normalized name.

This normalized representation will later be used to construct a blocking key.
The purpose of this first experiment is to establish a simple baseline before
testing more sophisticated blocking and matching strategies.

In [14]:
import re

def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    
    # Strip punctuation (& . , - etc.), keep letters/digits/spaces
    name = re.sub(r'[^\w\s]', ' ', name)
    
    # Remove common legal-suffix noise words
    stopwords = {'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
                 'corp', 'corporation', 'llc', 'llp', 'co', 'company'}
    tokens = [t for t in name.split() if t not in stopwords]
    
    # Collapse whitespace and return
    return ' '.join(tokens).strip()

In [15]:
sample_names = (
    train_source1['business_name'].head(10).tolist()
    + train_source2['business_name'].head(10).tolist()
    + train_source3['business_name'].head(10).tolist()
)

for n in sample_names:
    print(repr(n), "->", repr(normalize_name(n)))

"Orelee's Barbershop" -> 'orelee s barbershop'
'Prime Money' -> 'prime money'
'B+ Retail Inc' -> 'b retail'
'Christ Chapel' -> 'christ chapel'
'Prabhav Business Center' -> 'prabhav business center'
'Custom Wealth Services LLC' -> 'custom wealth services'
'Consulting Nyasa Nursing Private Limited' -> 'consulting nyasa nursing'
'Nexus Anchor Rain' -> 'nexus anchor rain'
'Moore Bitwise Inc' -> 'moore bitwise'
'Dermatology Green Medicine' -> 'dermatology green medicine'
'राम मार्केटिंग प्राइवेट लिमिटेड' -> 'र म म र क ट ग प र इव ट ल म ट ड'
'-- Holloway Peak Inc Seafood' -> 'holloway peak seafood'
'आदित्य प्रॉपर्टीज एलएलपी' -> 'आद त य प र पर ट ज एलएलप'
'Summit Inc' -> 'summit'
'Delta Tetlecommunication Inc' -> 'delta tetlecommunication'
'Lee and Lawson' -> 'lee and lawson'
'SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | www.shivshakti.com' -> 'shivshakti vidyalaya vidyalaya overseas www shivshakti com'
'Shree Infracon Private Ltd' -> 'shree infracon'
'Chavira Platinum Chimera LLC' -> 

### Normalization Sanity Check — Baseline

The normalization function was tested on a small sample of real business names
from all three sources.

For Latin-script names, the baseline behaves as expected in many cases. It
removes capitalization differences, punctuation, and common legal suffixes
while preserving the main business name.

However, the sample also revealed an important issue with non-Latin text.
Devanagari business names are being corrupted during punctuation removal. For
example, a Hindi name such as `राम मार्केटिंग प्राइवेट लिमिटेड` is transformed
into separated characters rather than preserving the original words.

This happens because the current regular expression does not correctly preserve
the Unicode combining characters used in Devanagari text.

Therefore, this version of the normalization function is not suitable as the
final baseline. We will create a new normalization experiment that handles
Unicode text more safely while preserving the useful normalization steps
identified here.

### 4.3 Experiment — Unicode-Safe Name Normalization

The previous normalization experiment showed that punctuation and legal-suffix
normalization works for many Latin-script names but breaks Devanagari text:
the `\w` regex character class does not recognize Devanagari vowel marks
(matras) as word characters, so they were stripped and words were shredded
into individual base letters.

In this experiment, we modify only the punctuation-handling step. Instead of
matching on `\w`, we remove characters using their Unicode general category
(punctuation categories only), which preserves letters and combining marks
across scripts while still stripping punctuation and normalizing whitespace.

The revised function is tested on the same sample as before so the two
normalization approaches can be compared directly.

In [18]:
import unicodedata

def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    
    # Remove only true punctuation characters (Unicode category starting with 'P'),
    # keep letters, combining marks (needed for Devanagari), digits, and spaces
    name = ''.join(' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch for ch in name)
    
    stopwords = {'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
                 'corp', 'corporation', 'llc', 'llp', 'co', 'company'}
    tokens = [t for t in name.split() if t not in stopwords]
    
    return ' '.join(tokens).strip()

In [19]:
sample_names = (
    train_source1['business_name'].head(10).tolist()
    + train_source2['business_name'].head(10).tolist()
    + train_source3['business_name'].head(10).tolist()
)

for n in sample_names:
    print(repr(n), "->", repr(normalize_name(n)))

"Orelee's Barbershop" -> 'orelee s barbershop'
'Prime Money' -> 'prime money'
'B+ Retail Inc' -> 'b retail'
'Christ Chapel' -> 'christ chapel'
'Prabhav Business Center' -> 'prabhav business center'
'Custom Wealth Services LLC' -> 'custom wealth services'
'Consulting Nyasa Nursing Private Limited' -> 'consulting nyasa nursing'
'Nexus Anchor Rain' -> 'nexus anchor rain'
'Moore Bitwise Inc' -> 'moore bitwise'
'Dermatology Green Medicine' -> 'dermatology green medicine'
'राम मार्केटिंग प्राइवेट लिमिटेड' -> 'राम मार्केटिंग प्राइवेट लिमिटेड'
'-- Holloway Peak Inc Seafood' -> 'holloway peak seafood'
'आदित्य प्रॉपर्टीज एलएलपी' -> 'आदित्य प्रॉपर्टीज एलएलपी'
'Summit Inc' -> 'summit'
'Delta Tetlecommunication Inc' -> 'delta tetlecommunication'
'Lee and Lawson' -> 'lee and lawson'
'SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | www.shivshakti.com' -> 'shivshakti vidyalaya vidyalaya overseas www shivshakti com'
'Shree Infracon Private Ltd' -> 'shree infracon'
'Chavira Platinum Chimera LLC' -

### 4.4 Blocking Key and Candidate Generation

After normalizing business names, we construct a simple blocking key to reduce
the number of Source 2 and Source 3 records that must be considered for each
Source 1 entity.

For this first blocking experiment, the key consists of the country and the
first token of the normalized business name. Using the entire normalized name
would be more restrictive and could discard true matches when names contain
typos or other variations. Using only the first token produces larger but more
forgiving candidate groups, while the country helps limit unrelated records.

We build lookup indexes for Source 2 and Source 3 so that candidates can be
retrieved efficiently without comparing every Source 1 entity against every
record in the other sources.

The blocking stage is evaluated on the held-out validation set using:
- candidate recall: the fraction of known true matches that survive blocking;
- average candidates per Source 1 entity: the size of the resulting candidate
  sets.

Candidate recall represents the recall ceiling of the downstream matching
stage: a true match that is removed during blocking cannot be recovered later.

In [20]:
def first_token(normalized_name):
    tokens = normalized_name.split()
    return tokens[0] if tokens else ""

def build_block_key(df):
    """Adds norm_name and block_key columns to a dataframe with business_name/country."""
    df = df.copy()
    df['norm_name'] = df['business_name'].apply(normalize_name)
    df['block_key'] = df['country'] + '||' + df['norm_name'].apply(first_token)
    return df

In [21]:
train_source2_keyed = build_block_key(train_source2)
train_source3_keyed = build_block_key(train_source3)
val_source1_keyed = build_block_key(val_source1)

print("Done building keys.")

Done building keys.


In [22]:
def build_index(df_keyed):
    """Returns dict: block_key -> list of entity_ids"""
    return df_keyed.groupby('block_key')['entity_id'].apply(list).to_dict()

index_source2 = build_index(train_source2_keyed)
index_source3 = build_index(train_source3_keyed)

print("Number of distinct keys in Source 2 index:", len(index_source2))
print("Number of distinct keys in Source 3 index:", len(index_source3))

Number of distinct keys in Source 2 index: 363912
Number of distinct keys in Source 3 index: 412262


In [23]:
def get_candidates(block_key):
    c2 = index_source2.get(block_key, [])
    c3 = index_source3.get(block_key, [])
    return set(c2) | set(c3)

val_gt_lookup = val_ground_truth.set_index('source1_entity_id')['matched_entity_ids']

recalls = []
candidate_counts = []

for _, row in val_source1_keyed.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    
    candidate_set = get_candidates(block_key)
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    
    candidate_counts.append(len(candidate_set))
    
    if len(true_set) == 0:
        # No true matches — recall undefined here, skip from recall average
        continue
    
    found = len(true_set & candidate_set)
    recalls.append(found / len(true_set))

print("Average candidates per entity:", sum(candidate_counts) / len(candidate_counts))
print("Candidate recall (macro-average, over entities with true matches):", sum(recalls) / len(recalls))
print("Entities with true matches evaluated:", len(recalls))

Average candidates per entity: 5680.185884698605
Candidate recall (macro-average, over entities with true matches): 0.7563608887127291
Entities with true matches evaluated: 416780


### Baseline Blocking Results

The first blocking experiment used the country and the first token of the
normalized business name as the blocking key.

On the 20% validation split, the Source 2 index contained 363,912 distinct
blocking keys and the Source 3 index contained 412,262 distinct keys.

The resulting candidate sets contained an average of 5,680.19 candidates per
Source 1 entity. Candidate recall was 75.64% across validation entities with at
least one known match.

This means that approximately 24% of true matches were excluded during
blocking and therefore could not be recovered by any downstream matching
model.

The baseline demonstrates that simple blocking can reduce the search space,
but the candidate sets are still large and the recall ceiling is not high
enough for the final pipeline. The next experiment will therefore investigate
a blocking strategy that improves recall without producing unnecessarily large
candidate sets.

In [24]:
# 1. Are a few keys responsible for most of the bloat? (bucket size distribution)
import numpy as np

bucket_sizes_s2 = [len(v) for v in index_source2.values()]
bucket_sizes_s3 = [len(v) for v in index_source3.values()]

print("Source 2 bucket sizes — mean:", np.mean(bucket_sizes_s2), "median:", np.median(bucket_sizes_s2), "max:", np.max(bucket_sizes_s2))
print("Source 3 bucket sizes — mean:", np.mean(bucket_sizes_s3), "median:", np.median(bucket_sizes_s3), "max:", np.max(bucket_sizes_s3))

Source 2 bucket sizes — mean: 13.83470729187276 median: 1.0 max: 40489
Source 3 bucket sizes — mean: 12.820980347448952 median: 1.0 max: 50862


In [25]:
# Inspect the largest blocking buckets

largest_s2 = sorted(
    index_source2.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:20]

largest_s3 = sorted(
    index_source3.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:20]

print("Largest Source 2 buckets:")
for key, ids in largest_s2:
    print(key, "->", len(ids))

print("\nLargest Source 3 buckets:")
for key, ids in largest_s3:
    print(key, "->", len(ids))

Largest Source 2 buckets:
US||the -> 40489
US||pediatric -> 30497
India||m -> 25796
India||shri -> 25702
India||dr -> 25673
India||sri -> 25658
India||smt -> 25474
India||mr -> 25449
India||new -> 14299
US||family -> 13231
US||a -> 11974
India||india -> 10655
US||eye -> 10532
US||ear -> 10393
India||mumbai -> 10384
US||blue -> 10379
US||physical -> 10338
US||dental -> 10315
US||foot -> 10272
US||primary -> 10221

Largest Source 3 buckets:
US||the -> 50862
US||pediatric -> 30944
India||m -> 29422
India||mr -> 29388
India||sri -> 29359
India||smt -> 29274
India||shri -> 29199
India||dr -> 29189
India||new -> 15783
US||family -> 13187
US||a -> 11972
India||india -> 10909
India||shree -> 10679
US||ear -> 10652
US||primary -> 10522
US||dental -> 10514
US||eye -> 10463
US||physical -> 10371
US||foot -> 10299
US||blue -> 10235


### Blocking Bucket Analysis

The first-token blocking strategy produced highly uneven bucket sizes. Although
the median bucket size was only 1 for both Source 2 and Source 3, a small number
of blocking keys produced extremely large buckets.

The largest buckets were associated with common or generic first tokens such as
`the`, `m`, `mr`, `dr`, `smt`, `sri`, and `shri`. For example, the largest Source
2 bucket contained 40,489 records for the key `US||the`, while the largest
Source 3 bucket contained 50,862 records for the same key.

This shows that the high average candidate count is driven disproportionately
by a relatively small number of very large buckets. It also suggests that
using the first token alone does not provide sufficiently selective blocking
for records beginning with common prefixes or generic terms.

We will next examine the true matches that were missed by this blocking
strategy before deciding how to modify the blocking key.

In [26]:
# 2. For entities where we LOST recall (missed a true match), was the true match's first word actually different?
# Sample a handful of failure cases to eyeball
sample_checked = 0
for _, row in val_source1_keyed.iterrows():
    entity_id = row['entity_id']
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    if len(true_set) == 0:
        continue
    candidate_set = get_candidates(row['block_key'])
    missed = true_set - candidate_set
    if missed:
        print("Source1:", row['business_name'], "| norm:", row['norm_name'])
        # look up one missed match's actual name
        missed_id = next(iter(missed))
        if missed_id.startswith('S2-'):
            match_row = train_source2[train_source2['entity_id'] == missed_id]
        else:
            match_row = train_source3[train_source3['entity_id'] == missed_id]
        print("  Missed match:", match_row['business_name'].values)
        sample_checked += 1
        if sample_checked >= 8:
            break

Source1: Prime Money | norm: prime money
  Missed match: ['primemoney.com']
Source1: Dream Construction Limited | norm: dream construction
  Missed match: ['డ్రీమ్ కన్\u200cస్ట్రక్షన్ లిమిటెడ్']
Source1: Rajani Enterprises of Kannur | norm: rajani enterprises of kannur
  Missed match: ['M/s Rajani Enterprises of']
Source1: Schaefer, Michael and Silver Associates | norm: schaefer michael and silver associates
  Missed match: ['Fayevio f/k/a Schaefer, Michael and Silver Associates']
Source1: Aggie E. Nagle, L.C.S.W. | norm: aggie e nagle l c s w
  Missed match: ['*** aggieenagle.com']
Source1: Pacific Learning Laboratories LLC | norm: pacific learning laboratories
  Missed match: ['... The Pacific Learning Laboratories [LLC] [LLC]']
Source1: Mayur Hospital of Gondal | norm: mayur hospital of gondal
  Missed match: ['M/s Mayur Hospital of Gondal Corporation']
Source1: Orchid Renewable Ltd | norm: orchid renewable
  Missed match: ['Smt BRIXUMBRABELO']


### Baseline Failure Analysis

The missed-match examples show that the first-token blocking strategy fails for
several different reasons. Some records contain leading noise prefixes such as
`M/s`, while others use alternative representations such as domain names or
different scripts. These cases indicate that improving blocking will require
more than simply changing the first-token rule.

We first address the clearly observed prefix and honorific noise because these
tokens also contributed to the largest blocking buckets. Cross-script and
other name-variation cases will be investigated separately in subsequent
experiments.

### 4.5 Experiment — Noise-Prefix Normalization

In [59]:
def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    name = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in name
    )
    
    stopwords = {
        'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
        'corp', 'corporation', 'llc', 'llp', 'co', 'company',
        'dr', 'mr', 'mrs', 'ms', 'smt', 'sri', 'shri', 'prof',
        # leading noise prefixes seen in real data
        'm', 's',  # "m/s" becomes "m s" after punctuation stripping
    }
    tokens = [t for t in name.split() if t not in stopwords]
    
    return ' '.join(tokens).strip()

In [60]:
train_source2_keyed_v2 = build_block_key(train_source2)
train_source3_keyed_v2 = build_block_key(train_source3)
val_source1_keyed_v2 = build_block_key(val_source1)

print("Done building v2 keys.")

Done building v2 keys.


In [61]:
index_source2_v2 = build_index(train_source2_keyed_v2)
index_source3_v2 = build_index(train_source3_keyed_v2)

print("Number of distinct keys in Source 2 index (v2):", len(index_source2_v2))
print("Number of distinct keys in Source 3 index (v2):", len(index_source3_v2))

Number of distinct keys in Source 2 index (v2): 372168
Number of distinct keys in Source 3 index (v2): 420329


In [62]:
bucket_sizes_s2_v2 = [len(v) for v in index_source2_v2.values()]
bucket_sizes_s3_v2 = [len(v) for v in index_source3_v2.values()]

print("Source 2 (v2) bucket sizes — mean:", np.mean(bucket_sizes_s2_v2), "median:", np.median(bucket_sizes_s2_v2), "max:", np.max(bucket_sizes_s2_v2))
print("Source 3 (v2) bucket sizes — mean:", np.mean(bucket_sizes_s3_v2), "median:", np.median(bucket_sizes_s3_v2), "max:", np.max(bucket_sizes_s3_v2))

Source 2 (v2) bucket sizes — mean: 13.527804647363556 median: 1.0 max: 40489
Source 3 (v2) bucket sizes — mean: 12.574918694641578 median: 1.0 max: 50863


In [ ]:
def get_candidates_v2(block_key):
    c2 = index_source2_v2.get(block_key, [])
    c3 = index_source3_v2.get(block_key, [])
    return set(c2) | set(c3)

recalls_v2 = []
candidate_counts_v2 = []

for _, row in val_source1_keyed_v2.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    
    candidate_set = get_candidates_v2(block_key)
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    
    candidate_counts_v2.append(len(candidate_set))
    
    if len(true_set) == 0:
        continue
    
    found = len(true_set & candidate_set)
    recalls_v2.append(found / len(true_set))

print("Average candidates per entity (v2):", sum(candidate_counts_v2) / len(candidate_counts_v2))
print("Candidate recall (v2):", sum(recalls_v2) / len(recalls_v2))
print("Entities with true matches evaluated:", len(recalls_v2))

### Experiment 4.5 Results — Noise-Prefix Normalization

The second normalization experiment removed common honorifics and leading noise
prefixes observed in the training data, including `dr`, `mr`, `mrs`, `ms`,
`smt`, `sri`, `shri`, `prof`, and the `m/s` representation after punctuation
normalization.

This modification increased candidate recall from 75.64% to 79.26%, an
improvement of approximately 3.63 percentage points.

The largest blocking buckets also became substantially smaller. The maximum
Source 2 bucket decreased from 40,489 to 30,875 records, while the maximum
Source 3 bucket decreased from 50,862 to 31,445 records.

However, the average candidate count increased slightly from 5,680.19 to
5,830.76 candidates per Source 1 entity. This indicates that removing the
prefixes redistributed records across blocking buckets rather than simply
reducing the overall candidate volume.

Overall, the normalization change improved the blocking recall and reduced the
largest bucket sizes, so this version will be retained for subsequent
experiments.

In [56]:
def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    name = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in name
    )
    
    stopwords = {
        'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
        'corp', 'corporation', 'llc', 'llp', 'co', 'company',
        'dr', 'mr', 'mrs', 'ms', 'smt', 'sri', 'shri', 'prof',
        'm', 's',
        'the'
    }
    tokens = [t for t in name.split() if t not in stopwords]
    
    return ' '.join(tokens).strip()

In [46]:
train_source2_keyed_v3 = build_block_key(train_source2)
train_source3_keyed_v3 = build_block_key(train_source3)
val_source1_keyed_v3 = build_block_key(val_source1)

In [58]:
index_source2_v3 = build_index(train_source2_keyed_v3)
index_source3_v3 = build_index(train_source3_keyed_v3)

print("Number of distinct keys in Source 2 index (v3):", len(index_source2_v3))
print("Number of distinct keys in Source 3 index (v3):", len(index_source3_v3))

Number of distinct keys in Source 2 index (v3): 372673
Number of distinct keys in Source 3 index (v3): 420923


In [57]:
bucket_sizes_s2_v3 = [len(v) for v in index_source2_v3.values()]
bucket_sizes_s3_v3 = [len(v) for v in index_source3_v3.values()]

print("Source 2 (v3) bucket sizes — mean:", np.mean(bucket_sizes_s2_v3), "median:", np.median(bucket_sizes_s2_v3), "max:", np.max(bucket_sizes_s2_v3))
print("Source 3 (v3) bucket sizes — mean:", np.mean(bucket_sizes_s3_v3), "median:", np.median(bucket_sizes_s3_v3), "max:", np.max(bucket_sizes_s3_v3))

Source 2 (v3) bucket sizes — mean: 13.509473452597854 median: 1.0 max: 30875
Source 3 (v3) bucket sizes — mean: 12.557173164688079 median: 1.0 max: 31445
